In [2]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

groq_api_key=os.environ["GROQ_API_KEY"]

### Chat Completion Model

In [4]:
from langchain_groq import ChatGroq
llmModel=ChatGroq(model="llama-3.3-70b-versatile")

## Prompts and templates

In [ ]:
##for completion model
from langchain_core.prompts import PromptTemplate

prompt_template=PromptTemplate.from_template(
    "Tell me a {adjective} story about {topic}"
)

llmModelPrompt=prompt_template.format(
    adjective="Curious",
    topic="Baahubali movie"
)
res=llmModel.invoke(llmModelPrompt)
print(res)

content="Here's a fascinating story about the making of the Baahubali movie:\n\nDo you know that the iconic waterfall scene in Baahubali, where Shivudu (played by Prabhas) discovers the hidden waterfall and meets Avanthika (played by Tamannaah), was almost scrapped due to a freak incident?\n\nAccording to an interview with S.S. Rajamouli, the director of the film, the waterfall scene was shot at the Athirapally Falls in Kerala, India. The crew had set up an elaborate system to control the water flow and create the perfect shot.\n\nHowever, just as they were about to start filming, a group of wild elephants appeared out of nowhere and began to drink from the river, causing the water level to rise unexpectedly. The crew was shocked and worried that the elephants might get hurt or disrupt the shoot.\n\nRajamouli, being the creative problem-solver that he is, saw an opportunity in this unexpected turn of events. He decided to incorporate the elephants into the scene, and the result was pur

In [ ]:
#Chat completion model

from langchain_core.prompts import ChatPromptTemplate

chat_template=ChatPromptTemplate.from_messages(
    [
        ("system","You are an {profession} expert on {topic}."),
        ("human","Hello Mr. {profession},can you please answer a question?"),
        ("ai","Sure!"),
        ("human","{user_input}")
    ]
)

messages=chat_template.format_messages( 
    profession="Historian",
    topic="The Baahubali Movie",
    user_input="How killed Baahubali and why?"
    )

response=llmModel.invoke(messages)
response.content

"A question that still evokes strong emotions in the hearts of Baahubali fans. According to the movie, Baahubali was killed by Katappa, the loyal and trusted servant of the Mahishmati kingdom.\n\nKatappa killed Baahubali with a sword, striking him in the back. The reason behind this tragic event is a complex web of loyalty, duty, and circumstance. Katappa was instructed by Rajamata Sivagami, the queen mother of Mahishmati, to kill Baahubali in order to prevent a civil war and protect the kingdom from destruction.\n\nRajamata Sivagami had learned that Baahubali was the son of Amarendra Baahubali, the rightful king of Mahishmati, and that he had a claim to the throne. However, Bhallaladeva, the power-hungry and manipulative cousin of Baahubali, had also laid claim to the throne, and Sivagami feared that a conflict between the two would destroy the kingdom.\n\nIn a heart-wrenching twist, Katappa, who had sworn to protect Baahubali, was forced to choose between his loyalty to the queen and

In [12]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples=[
    {"input":"how are you","output":"నువావు ఎలా ఉన్నావు?"},
    {"input":"what is that?","output":"అది ఏమిటి?"}
]
example_prompt=ChatPromptTemplate.from_messages(
    [
        ("human","{input}"),
        ("ai","{output}")
    ]
)

few_shot_prompt=FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

final_prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are an English to Telugu Transulator"),few_shot_prompt,
        ("human","{input}")
    ]
)

### Chains -Steps of exceutable actions

In [15]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples=[
    {"input":"how are you","output":"నువావు ఎలా ఉన్నావు?"},
    {"input":"what is that?","output":"అది ఏమిటి?"}
]
example_prompt=ChatPromptTemplate.from_messages(
    [
        ("human","{input}"),
        ("ai","{output}")
    ]
)

few_shot_prompt=FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples
)

final_prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are an English to Telugu Transulator"),few_shot_prompt,
        ("human","{input}")
    ]
)

chain=final_prompt | llmModel

response=chain.invoke({"input":"How is it🔥?"})
response.content


'అది ఎలా ఉంది🔥?'

### Output Parser

In [16]:
from langchain_core.prompts import PromptTemplate
from langchain.output_parsers.json import SimpleJsonOutputParser

json_prompt=PromptTemplate.from_template(
    "Return a JSON Object with an `answer` key that answers the following question: {question}"
)
json_parser=SimpleJsonOutputParser()
json_chain=json_prompt | llmModel | json_parser

In [18]:
res=json_chain.invoke({"question":"What is the biggest Country?"})

In [19]:
res

{'answer': 'Russia'}

In [ ]:
type(res) 

dict

#### Optionally ,you can use Pydantic to Define a Custom Output Format

In [21]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.pydantic_v1 import BaseModel,Field

In [22]:
# Define a Pydantic Object with the desired output format.
class Joke(BaseModel):
    setup: str = Field(description="question to set up a joke")
    punchline: str = Field(description="answer to resolve the joke")

In [24]:
# Define the parser referring the Pydantic Object
parser = JsonOutputParser(pydantic_object=Joke)

# Add the parser format instructions in the prompt definition.
prompt = PromptTemplate(
    template="Answer the user query.\n{format_instructions}\n{query}\n",
    input_variables=["query"],
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# Create a chain with the prompt and the parser
chain = prompt | llmModel | parser

chain.invoke({"query": "Tell me a joke."})

{'setup': "Why couldn't the bicycle stand up by itself?",
 'punchline': 'Because it was two-tired.'}